### Positional Embedding 

why do we need positional embedding if we have token embedding?

ex - 1. cat is on the mat .
     2. mat is on the cat . 

token embedding of the cat is same in both the sentences.
but the position of the word is different due to which meaning is changed. so , how can infuse the information about the position so the model can understand overall context this is done by -> positional embedding


### 1. Absolute positional embedding

1. cat ---> cat_token_ID = 456

2. cat_token_ID--->cat_token_embedding = [0.43,0.22,0.54]

3. cat_token_embedding ------> cat_positional_embedding  =  cat_token_embedding + position_vector

4. cat_positional_embedding. =. [0.43,0.22,0.54]+[1.2,1.3,1.4]

positional vector dim = token embedding vector



In [79]:
import torch

## 1. row data

In [81]:
with open("the-verdict-book.txt","r",encoding="utf-8") as f:
    raw_text=f.read()
raw_text[:22]

'I HAD always thought J'

In [82]:
! pip install tiktoken 
import tiktoken

## 2.implement data loader and dataset

In [83]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text,embedding, context_size, stride):
        self.input_id = []
        self.target_id = []

         # tokenize the entire text 
        token_id= embedding.encode(text)

        # using sliding window approach to create input and target pairs
        for i in range(0, len(token_id) - context_size, stride):
          input_token = token_id[i:i+context_size]
          target_token = token_id[i+1:i+context_size+1]
          self.input_id.append(torch.tensor(input_token))
          self.target_id.append(torch.tensor(target_token))

    def __len__(self):
        return len(self.input_id)

    def __getitem__(self, idx):
        return self.input_id[idx], self.target_id[idx]

In [84]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [85]:
## 3. inputs - target pairs

In [86]:

max_length=4
batch_size=8
stride=4

dataloader=create_dataloader_v1(raw_text,batch_size=batch_size,
                                max_length=max_length,
                                stride=stride,shuffle=False)

data_iter=iter(dataloader)
input,target=next(data_iter)
print("token ID :\n",input)
print("input shape :\n",input.shape)


token ID :
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
input shape :
 torch.Size([8, 4])


## 4.make embedding matrix

In [87]:
vocab = 50257
vector_dim=256
torch.manual_seed(123)
embedding_matrix=torch.nn.Embedding(vocab,vector_dim)
print(embedding_matrix.weight)


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035,  ...,  1.3337,  0.0771, -0.0522],
        [ 0.2386,  0.1411, -1.3354,  ..., -0.0315, -1.0640,  0.9417],
        [-1.3152, -0.0677, -0.1350,  ..., -0.3181, -1.3936,  0.5226],
        ...,
        [ 0.5871, -0.0572, -1.1628,  ..., -0.6887, -0.7364,  0.4479],
        [ 0.4438,  0.7411,  1.1263,  ...,  1.2091,  0.6781,  0.3331],
        [-0.2537,  0.1446,  0.7203,  ..., -0.2134,  0.2144,  0.3006]],
       requires_grad=True)


In [88]:
input[0]

tensor([  40,  367, 2885, 1464])

In [89]:
input[0][0]

tensor(40)

In [90]:
print(embedding_matrix(input[0][0]).shape)

torch.Size([256])


In [91]:
token_embedding=embedding_matrix(input)
token_embedding.shape

torch.Size([8, 4, 256])

## 5. positional embedding

In [92]:
context_length=max_length
pos_vector=torch.nn.Embedding(context_length,vector_dim)
pos_embeddings = pos_vector(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [93]:
input_embedding=token_embedding+pos_embeddings
input_embedding.shape

torch.Size([8, 4, 256])

### 2. Relative position embedding

use for the relative postion.

GPT and all are use the absloute embedding